# Task 2

## Text Chunking, Embedding and Vector Store

This notebook demonstrates:

- Stratified sampling
- Text chunking
- Sentence embeddings
- ChromaDB vector database creation

In [3]:
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer

from langchain_text_splitters import RecursiveCharacterTextSplitter

from tqdm import tqdm

tqdm.pandas()

## Load Cleaned Dataset

In [4]:
df = pd.read_csv("../data/processed/filtered_complaints.csv")

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,word_count,clean_text
0,2025-06-13,Credit card,Store credit card,Getting a credit card,Card opened without my consent or knowledge,A XXXX XXXX card was opened under my name by a...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78230,Servicemember,Consent provided,Web,2025-06-13,Closed with non-monetary relief,Yes,NaN,14069121,91,a xxxx xxxx card was opened under my name by a...
1,2025-06-12,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Other problem,"Dear CFPB, I have a secured credit card with c...",Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11220,NaN,Consent provided,Web,2025-06-13,Closed with monetary relief,Yes,NaN,14047085,156,dear cfpb i have a secured credit card with ci...
2,2025-06-12,Credit card,General-purpose credit card or charge card,Incorrect information on your report,Account information incorrect,I have a Citi rewards cards. The credit balanc...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",IL,60067,NaN,Consent provided,Web,2025-06-12,Closed with explanation,Yes,NaN,14040217,233,i have a citi rewards cards the credit balance...
3,2025-06-09,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,b'I am writing to dispute the following charge...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",TX,78413,Older American,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13968411,454,b i am writing to dispute the following charge...
4,2025-06-09,Credit card,General-purpose credit card or charge card,Problem when making payments,Problem during payment process,"Although the account had been deemed closed, I...",Company believes it acted appropriately as aut...,Atlanticus Services Corporation,NY,11212,Older American,Consent provided,Web,2025-06-09,Closed with monetary relief,Yes,NaN,13965746,170,although the account had been deemed closed i ...


## Product Distribution

In [5]:
df["Product"].value_counts()

Product
Credit card    80667
Name: count, dtype: int64

## Stratified Sampling

In [6]:
sample_size = 12000

In [12]:
sample_df = (
    df
    .groupby("Product", group_keys=True)
    .apply(
        lambda x: x.sample(
            int(sample_size * len(x) / len(df)),
            random_state=42
        )
    )
    .reset_index(drop=False)
)

In [13]:
sample_df.shape

(12000, 21)

In [14]:
sample_df["Product"].value_counts()

Product
Credit card    12000
Name: count, dtype: int64

In [15]:
sample_df.to_csv(
    "../data/processed/sample_complaints.csv",
    index=False
)

## Text Chunking

In [17]:
import sys
sys.path.append("..")

from src.chunking import chunk_text

In [18]:
sample_df["chunks"] = sample_df["clean_text"].progress_apply(chunk_text)

100%|██████████| 12000/12000 [00:06<00:00, 1955.73it/s]


In [19]:
sample_df.head()

,Product,level_1,Date received,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,...,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,word_count,clean_text,chunks
0,Credit card,60999,2023-09-14,Store credit card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,I have a balance of {$710.00} from my XXXX XXX...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",IL,...,Consent provided,Web,2023-09-14,Closed with explanation,Yes,NaN,7548263,150,i have a balance of 710 00 from my xxxx xxxx x...,[i have a balance of 710 00 from my xxxx xxxx ...
1,Credit card,6927,2025-01-28,General-purpose credit card or charge card,Fees or interest,Unexpected increase in interest rate,My original credit card rate on my American Ex...,NaN,AMERICAN EXPRESS COMPANY,SC,...,Consent provided,Web,2025-01-31,Closed with explanation,Yes,NaN,11805170,96,my original credit card rate on my american ex...,[my original credit card rate on my american e...
2,Credit card,34733,2024-09-07,General-purpose credit card or charge card,Improper use of your report,Credit inquiries on your report that you don't...,XXXX XXXX XXXX XXXX XXXX XXXX XXXX FL XXXX TRA...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",FL,...,Consent provided,Web,2024-09-07,Closed with explanation,Yes,NaN,10034123,117,xxxx xxxx xxxx xxxx xxxx xxxx xxxx fl xxxx tra...,[xxxx xxxx xxxx xxxx xxxx xxxx xxxx fl xxxx tr...
3,Credit card,44531,2016-08-03,NaN,Other,NaN,GENDER DISCRIMINATION - I used a kiosk at XXXX...,Company has responded to the consumer and the ...,SYNCHRONY FINANCIAL,MA,...,Consent provided,Web,2016-08-03,Closed with explanation,Yes,No,2044150,302,gender discrimination i used a kiosk at xxxx x...,[gender discrimination i used a kiosk at xxxx ...
4,Credit card,58684,2023-11-03,General-purpose credit card or charge card,Getting a credit card,Card opened without my consent or knowledge,JPMB J.P Morgan hard inquiry was opened on my ...,NaN,JPMORGAN CHASE & CO.,GA,...,Consent provided,Web,2023-11-09,Closed with non-monetary relief,Yes,NaN,7799737,29,jpmb j p morgan hard inquiry was opened on my ...,[jpmb j p morgan hard inquiry was opened on my...


In [20]:
records = []

for _, row in sample_df.iterrows():

    for idx, chunk in enumerate(row["chunks"]):

        records.append({

            "complaint_id": row.get("Complaint ID", row.name),

            "product": row["Product"],

            "chunk_index": idx,

            "text": chunk

        })

chunks_df = pd.DataFrame(records)

In [21]:
chunks_df.head()

,complaint_id,product,chunk_index,text
0,7548263,Credit card,0,i have a balance of 710 00 from my xxxx xxxx x...
1,7548263,Credit card,1,support wayfair directed me to xxxx and xxxx d...
2,11805170,Credit card,0,my original credit card rate on my american ex...
3,11805170,Credit card,1,by american express that is contrary to the te...
4,10034123,Credit card,0,xxxx xxxx xxxx xxxx xxxx xxxx xxxx fl xxxx tra...


In [22]:
chunks_df.shape

(34054, 4)

## Embedding Model

In [23]:
import sys
sys.path.append("..")

from src.embeddings import EmbeddingModel

In [24]:
embedding_model = EmbeddingModel()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\sumex\Desktop\rag-complaint-chatbot\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sumex\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [25]:
embeddings = embedding_model.embed(
    chunks_df["text"].tolist()
)

Batches:   0%|          | 0/1065 [00:00<?, ?it/s]

In [26]:
len(embeddings)

34054

In [27]:
import sys
sys.path.append("..")

from vector_store.vector_store import ComplaintVectorStore

In [28]:
store = ComplaintVectorStore()

In [29]:
ids = []

documents = []

metadatas = []

for i, row in chunks_df.iterrows():

    ids.append(str(i))

    documents.append(row["text"])

    metadatas.append({

        "complaint_id": str(row["complaint_id"]),

        "product": row["product"],

        "chunk_index": row["chunk_index"]

    })

In [31]:
batch_size = 5000  # must be ≤ 5461
for start in range(0, len(ids), batch_size):
    end = start + batch_size
    store.add(
        ids[start:end],
        documents[start:end],
        embeddings[start:end],
        metadatas[start:end]
    )


In [32]:
print("Vector store created successfully.")

Vector store created successfully.


In [33]:
chunks_df.to_csv(
    "../data/processed/chunks.csv",
    index=False
)

## Task 2 Summary

A stratified sample of approximately 12,000 complaints was selected to ensure proportional representation across the four product categories. This approach reduced computational requirements while preserving the underlying complaint distribution.

Complaint narratives were split into overlapping text chunks using LangChain's RecursiveCharacterTextSplitter with a chunk size of 500 characters and an overlap of 50 characters. This configuration balances contextual completeness with retrieval accuracy by minimizing the risk of splitting important information across chunk boundaries.

Text embeddings were generated using the sentence-transformers/all-MiniLM-L6-v2 model. This lightweight model produces 384-dimensional embeddings and is widely adopted for semantic similarity tasks because it offers a strong trade-off between retrieval quality, speed, and memory usage.

The resulting embeddings, together with complaint metadata, were stored in a persistent ChromaDB vector database. Although the project later uses the provided full-scale vector store for inference, constructing a sample vector database demonstrates the complete indexing workflow required for Retrieval-Augmented Generation systems.